# Model Hasil Riset Fundamental Reguler Terintegrasi: Custom CNN vs MobileNetV2 vs VGG16 vs YOLOv8-cls
## Klasifikasi Biner Limbah Plastik Pesisir — Citra Sentinel-2A True-Color
### Versi Final: Single Split + Bootstrap 95% CI + McNemar's Test

**Notebook ini** memakai skema evaluasi **single train/val split** (seperti pipeline asli Anda -- bukan 5-fold CV), tapi diperkuat dengan dua teknik validasi statistik yang TIDAK memerlukan retraining tambahan:

1. **Bootstrap resampling (2000x)** pada 77 prediksi hasil evaluasi -- menghasilkan 95% confidence interval untuk akurasi, F1, dan Kappa tiap model. Ini menjawab pertanyaan "seberapa stabil angka ini kalau sampel ujinya sedikit berbeda?" tanpa perlu retrain model sama sekali.
2. **McNemar's test** antar tiap pasangan model -- uji statistik formal untuk membuktikan apakah perbedaan performa antar model (termasuk kasus MobileNetV2 vs YOLOv8-cls yang identik) signifikan secara statistik atau bisa jadi kebetulan.

Kedua teknik ini jauh lebih ringan secara komputasi dibanding 5-fold cross-validation penuh (tidak perlu GPU kuat / Colab Pro), sambil tetap memberi bukti statistik yang lebih kuat dibanding satu angka akurasi mentah.

**Struktur notebook:**
1. Setup & Konfigurasi
2. Load & Validasi Dataset
3. Konversi TIF → PNG
4. Definisi Fungsi Model
5. Split Data (single split, seperti pipeline asli)
6. Training & Evaluasi 4 Model
7. Tabel Perbandingan Akhir
8. **BARU:** Bootstrap 95% CI + McNemar's Test
9. Ringkasan Siap-Tempel untuk Artikel

## 1. Setup & Konfigurasi

In [ ]:
# ==========================================
# IMPORTS
# ==========================================
import os
import sys
import shutil
import logging
from pathlib import Path

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models, regularizers
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.applications import MobileNetV2, VGG16
    from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
    from tensorflow.keras.models import Model
    from tensorflow.keras.optimizers import Adam
except ImportError as e:
    raise ImportError(
        f"TensorFlow/Keras tidak ditemukan atau gagal diimpor: {e}. "
        f"Install dengan: pip install tensorflow --break-system-packages"
    ) from e

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
from sklearn.utils.class_weight import compute_class_weight


def setup_logging():
    """Logger konsisten untuk seluruh pipeline -- tampil di console + tersimpan ke file."""
    logger = logging.getLogger("plastik_pipeline")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("[%(asctime)s] %(levelname)s: %(message)s", datefmt="%H:%M:%S")

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    try:
        fh = logging.FileHandler("pipeline_log.txt", mode="a", encoding="utf-8")
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    except OSError as e:
        print(f"PERINGATAN: gagal membuat file log: {e}. Lanjut dengan console logging saja.")

    return logger

logger = setup_logging()
logger.info("Logger siap. Memulai pipeline training terintegrasi (versi 5-fold CV).")

In [ ]:
# ==========================================
# MOUNT GOOGLE DRIVE (khusus Colab) -- aman dijalankan berulang
# ==========================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    logger.info("Google Drive berhasil di-mount.")
except ImportError:
    logger.warning(
        "Modul 'google.colab' tidak ditemukan -- sepertinya bukan berjalan di Colab. "
        "Pastikan path CONFIG di bawah disesuaikan dengan environment lokal Anda."
    )
except Exception as e:
    logger.error(f"Gagal mount Google Drive: {e}")
    raise

In [ ]:
# ==========================================
# CEK GPU -- WARNING JELAS JIKA TIDAK ADA GPU
# ==========================================
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    logger.info(f"GPU terdeteksi: {gpu_devices}")
else:
    logger.warning(
        "TIDAK ADA GPU terdeteksi (GPU Available: []). "
        "Training akan SANGAT LAMBAT, terutama untuk MobileNetV2 dan proses 5-fold CV. "
        "Di Colab: Runtime > Change runtime type > pilih GPU (T4), lalu restart runtime."
    )

In [ ]:
import os  # Added for os.makedirs

# ==========================================
# KONFIGURASI GLOBAL -- SESUAIKAN DENGAN SETUP AHMAD
# ==========================================
CONFIG = {
    # --- Path & data ---
    "csv_path": "/content/drive/MyDrive/project_BIMA26/manifest.csv/layer_gabungfix_parebarrumakassar28426.csv",
    "images_dir_raw": "/content/drive/MyDrive/project_BIMA26/outputgbg_paremakassar28426_png_rasterio",
    "images_dir_png": "/content/drive/MyDrive/project_BIMA26/outputgbg_paremakassar28426_png_rasterio",
    "x_col": "Patch_ID",
    "y_col": "Jenis_Obyek",
    "class_names": ["non_plastik", "plastik"],
    "minority_class": "plastik",

    # --- Split (tidak lagi dipakai untuk single-split, dipertahankan untuk kompatibilitas) ---
    "test_size": 0.2,
    "random_state": 42,

    # --- Training umum (Custom CNN & MobileNetV2) ---
    "img_size": (32, 32),
    "batch_size": 16,
    "epochs": 50,

    # --- YOLOv8-cls ---
    "yolo_dataset_dir": "/content/drive/MyDrive/project_BIMA26/yolov8_dataset_integrated",
    "yolo_model_arch": "yolov8n-cls.pt",
    "yolo_epochs": 60,
    "yolo_imgsz": 32,
    "yolo_batch": 16,
    "yolo_lr0": 0.001,
    "yolo_lrf": 0.01,
    "yolo_patience": 15,

    # --- Output ---
    "save_dir": "/content/drive/MyDrive/project_BIMA26/model_comparison_outputs",
}

os.makedirs(CONFIG["save_dir"], exist_ok=True)

# Reproduksibilitas: fix random seed TensorFlow & numpy (selain random_state sklearn yang sudah ada)
tf.random.set_seed(CONFIG["random_state"])
np.random.seed(CONFIG["random_state"])

logger.info("CONFIG berhasil didefinisikan. Sesuaikan path di atas jika struktur folder Anda berbeda.")

## 2. Load & Validasi Dataset (CSV + Gambar)

In [ ]:
import pandas as pd
import os
import logging

def load_and_validate_csv(config: dict, logger: logging.Logger) -> pd.DataFrame:
    """
    Load CSV dan validasi kolom wajib + normalisasi ekstensi file ke .png.

    Raises:
        FileNotFoundError: jika file CSV tidak ditemukan
        KeyError: jika kolom X_COL / Y_COL tidak ada di CSV
        ValueError: jika CSV kosong atau class_names tidak cocok dengan isi Y_COL
    """
    csv_path = config["csv_path"]
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"File CSV tidak ditemukan di '{csv_path}'. "
            f"Periksa kembali path di CONFIG['csv_path'] dan pastikan Drive sudah ter-mount."
        )

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        raise ValueError(f"Gagal membaca CSV '{csv_path}': {type(e).__name__}: {e}") from e

    if df.empty:
        raise ValueError(f"CSV '{csv_path}' berhasil dibaca tapi kosong (0 baris).")

    x_col, y_col = config["x_col"], config["y_col"]
    missing_cols = [c for c in (x_col, y_col) if c not in df.columns]
    if missing_cols:
        raise KeyError(
            f"Kolom wajib {missing_cols} tidak ditemukan di CSV. "
            f"Kolom yang tersedia: {df.columns.tolist()}. "
            f"Perbarui CONFIG['x_col']/CONFIG['y_col'] agar sesuai."
        )

    # Normalisasi ekstensi .tif -> .png di kolom nama file
    df[x_col] = df[x_col].apply(
        lambda f: f[:-4] + '.png' if isinstance(f, str) and f.lower().endswith('.tif') else f
    )

    unexpected_classes = set(df[y_col].unique()) - set(config["class_names"])
    if unexpected_classes:
        raise ValueError(
            f"Ditemukan kelas tak terduga di kolom '{y_col}': {unexpected_classes}. "
            f"Kelas yang diharapkan: {config['class_names']}. "
            f"Perbarui CONFIG['class_names'] atau periksa data mentah."
        )

    logger.info(f"CSV berhasil dimuat: {len(df)} baris.")
    logger.info(f"Distribusi kelas:\n{df[y_col].value_counts().to_string()}")
    return df


def validate_images_exist(df: pd.DataFrame, images_dir: str, x_col: str, logger: logging.Logger,
                            sample_check: bool = True) -> None:
    """
    Validasi bahwa direktori gambar ada dan berisi file yang direferensikan CSV.
    Jika sample_check=True, hanya cek subset acak (lebih cepat untuk dataset besar).

    Raises:
        FileNotFoundError: jika direktori tidak ada, atau proporsi file hilang terlalu tinggi
    """
    if not os.path.isdir(images_dir):
        raise FileNotFoundError(
            f"Direktori gambar tidak ditemukan: '{images_dir}'. "
            f"Periksa CONFIG['images_dir_raw'] / CONFIG['images_dir_png']."
        )

    check_df = df.sample(min(50, len(df)), random_state=42) if sample_check else df
    missing = [f for f in check_df[x_col] if not os.path.exists(os.path.join(images_dir, f))]

    missing_ratio = len(missing) / len(check_df)
    if missing_ratio > 0.5:
        raise FileNotFoundError(
            f"{len(missing)}/{len(check_df)} file gambar sampel TIDAK ditemukan di '{images_dir}'. "
            f"Contoh file hilang: {missing[:5]}. "
            f"Kemungkinan: (1) ekstensi file belum dikonversi (masih .tif), "
            f"(2) IMAGES_DIR menunjuk ke folder yang salah."
        )
    elif missing:
        logger.warning(f"{len(missing)}/{len(check_df)} file sampel tidak ditemukan -- lanjut, tapi periksa data.")
    else:
        logger.info(f"Validasi gambar OK: sampel {len(check_df)} file semuanya ditemukan di '{images_dir}'.")


df = load_and_validate_csv(CONFIG, logger)
validate_images_exist(df, CONFIG["images_dir_raw"], CONFIG["x_col"], logger)

### Distribusi Kelas `Jenis_Obyek`

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x=CONFIG['y_col'], hue=CONFIG['y_col'], palette='viridis', legend=False)
plt.title(f'Distribusi Kelas {CONFIG["y_col"]}')
plt.xlabel(CONFIG['y_col'])
plt.ylabel('Jumlah Sampel')
plt.show()

## 3. Konversi TIF → PNG (Idempoten)

Fungsi ini **aman dijalankan berkali-kali** -- file yang sudah dikonversi otomatis dilewati.

In [ ]:
import os
import numpy as np
import pandas as pd
import logging

def convert_tif_to_png(images_dir_raw: str, images_dir_png: str, df: pd.DataFrame, x_col: str,
                        logger: logging.Logger) -> str:
    """
    Konversi .tif -> .png satu kali, skip file yang sudah ada.
    Coba pakai rasterio dulu (lebih robust untuk GeoTIFF multiband), fallback ke PIL.

    Returns:
        str: path direktori PNG final (untuk update CONFIG['images_dir_png'])

    Raises:
        RuntimeError: jika SEMUA file gagal dikonversi (bukan sebagian)
    """
    os.makedirs(images_dir_png, exist_ok=True)

    try:
        import rasterio
        use_rasterio = True
    except ImportError:
        logger.warning("Library 'rasterio' tidak tersedia, fallback ke PIL untuk konversi.")
        use_rasterio = False

    from PIL import Image

    converted, skipped, failed = 0, 0, 0
    failed_files = []

    for _, row in df.iterrows():
        raw_name = row[x_col]
        if not isinstance(raw_name, str):
            continue
        tif_name = raw_name.replace('.png', '.tif') if raw_name.lower().endswith('.png') else raw_name
        png_name = tif_name.replace('.tif', '.png') if tif_name.lower().endswith('.tif') else tif_name

        src_path = os.path.join(images_dir_raw, tif_name)
        dst_path = os.path.join(images_dir_png, png_name)

        if os.path.exists(dst_path):
            skipped += 1
            continue
        if not os.path.exists(src_path):
            failed += 1
            failed_files.append(tif_name)
            continue

        try:
            if use_rasterio:
                with rasterio.open(src_path) as src:
                    arr = src.read()
                    arr = np.transpose(arr, (1, 2, 0)) if arr.ndim == 3 else arr
                    if arr.dtype != np.uint8:
                        arr = ((arr - arr.min()) / (arr.max() - arr.min() + 1e-9) * 255).astype(np.uint8)
                    Image.fromarray(arr[:, :, :3] if arr.shape[-1] >= 3 else arr.squeeze()).save(dst_path, 'PNG')
            else:
                with Image.open(src_path) as img:
                    img.save(dst_path, 'PNG')
            converted += 1
        except Exception as e:
            failed += 1
            failed_files.append(f"{tif_name} ({type(e).__name__}: {e})")

    logger.info(f"Konversi TIF->PNG selesai. Berhasil: {converted}, sudah ada (skip): {skipped}, gagal: {failed}")
    if failed_files:
        logger.warning(f"Contoh file gagal/hilang: {failed_files[:5]}")

    if converted == 0 and skipped == 0:
        raise RuntimeError(
            f"SEMUA {failed} file gagal dikonversi. Periksa apakah '{images_dir_raw}' berisi file yang benar."
        )

    return images_dir_png


# Jalankan konversi hanya jika folder PNG target belum berisi cukup file
_png_dir = CONFIG["images_dir_png"]
_existing_png_count = len([f for f in os.listdir(_png_dir) if f.lower().endswith('.png')]) if os.path.isdir(_png_dir) else 0

if _existing_png_count >= len(df) * 0.9:
    logger.info(f"Folder PNG sudah berisi {_existing_png_count} file (>=90% dataset) -- lewati konversi.")
else:
    CONFIG["images_dir_png"] = convert_tif_to_png(
        CONFIG["images_dir_raw"], CONFIG["images_dir_png"], df, CONFIG["x_col"], logger
    )

# Pastikan kolom X_COL di df memakai ekstensi .png untuk semua tahap selanjutnya
df[CONFIG["x_col"]] = df[CONFIG["x_col"]].str.replace('.tif', '.png', regex=False)
IMAGES_DIR = CONFIG["images_dir_png"]

## 4. Definisi Fungsi Model & Evaluasi

Sel-sel di bawah ini HANYA mendefinisikan fungsi (tidak langsung dieksekusi dengan split tunggal) --
eksekusi sesungguhnya terjadi di loop 5-fold cross-validation pada bagian 6.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

def build_generators(df_train: pd.DataFrame, df_val: pd.DataFrame, images_dir: str, config: dict,
                      logger: logging.Logger):
    """
    Bangun ImageDataGenerator untuk training (dengan augmentasi) dan validasi (tanpa augmentasi).
    Augmentasi dibuat KONSERVATIF karena citra true-color -- perubahan warna berlebihan
    berisiko merusak sinyal spektral yang jadi dasar klasifikasi plastik vs non-plastik.

    Raises:
        RuntimeError: jika generator gagal menemukan gambar sama sekali (0 sampel)
    """
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        rotation_range=30,
        width_shift_range=0.15,
        height_shift_range=0.15,
        shear_range=0.2,
        zoom_range=0.15,
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode='nearest',
    )
    val_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

    try:
        train_generator = train_datagen.flow_from_dataframe(
            dataframe=df_train, directory=images_dir,
            x_col=config["x_col"], y_col=config["y_col"],
            target_size=config["img_size"], batch_size=config["batch_size"],
            class_mode='binary', classes=config["class_names"], shuffle=True,
            seed=config["random_state"],
        )
        val_generator = val_datagen.flow_from_dataframe(
            dataframe=df_val, directory=images_dir,
            x_col=config["x_col"], y_col=config["y_col"],
            target_size=config["img_size"], batch_size=config["batch_size"],
            class_mode='binary', classes=config["class_names"], shuffle=False,
        )
    except Exception as e:
        raise RuntimeError(
            f"Gagal membangun ImageDataGenerator: {type(e).__name__}: {e}. "
            f"Periksa apakah 'images_dir' benar dan file gambar bisa dibuka."
        ) from e

    if train_generator.samples == 0 or val_generator.samples == 0:
        raise RuntimeError(
            f"Generator berhasil dibuat tapi 0 sampel ditemukan "
            f"(train={train_generator.samples}, val={val_generator.samples}). "
            f"Kemungkinan nama file di CSV tidak cocok dengan file di '{images_dir}'."
        )

    logger.info(f"Generator siap: train={train_generator.samples} sampel, val={val_generator.samples} sampel.")
    return train_generator, val_generator


def compute_class_weights_dict(train_generator, logger: logging.Logger) -> dict:
    """
    Hitung class_weight untuk mengatasi ketidakseimbangan kelas sisa (setelah oversampling).
    """
    y_train_labels = train_generator.classes
    unique_classes = np.unique(y_train_labels)
    weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=y_train_labels)
    class_weights_dict = dict(zip(unique_classes, weights))
    logger.info(f"Class weights: {class_weights_dict}")
    return class_weights_dict

In [ ]:
import sys
import traceback

def build_custom_cnn(input_shape, learning_rate=1e-3):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model


def train_keras_model(model, model_name: str, train_generator, val_generator, config: dict,
                       class_weights_dict: dict, logger: logging.Logger, patience: int = 10):
    """
    Training generik untuk model Keras (Custom CNN / MobileNetV2 / VGG16)
    dengan callbacks EarlyStopping + ReduceLROnPlateau, dibungkus try-except.

    `patience` bisa dikustomisasi per pemanggilan -- Custom CNN (dilatih from scratch, paling
    rentan terhenti terlalu awal di dataran rendah) diberi patience lebih besar dari default.

    Returns:
        History object jika training berhasil, None jika gagal.
    """
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=0),
    ]

    logger.info(f"Memulai training '{model_name}' -- epochs={config['epochs']}, batch={config['batch_size']}")
    history = None
    try:
        history = model.fit(
            train_generator, epochs=config["epochs"], validation_data=val_generator,
            callbacks=callbacks, class_weight=class_weights_dict,
            verbose=0
        )
    except tf.errors.ResourceExhaustedError as e:
        error_msg = f"--- ERROR: [{model_name}] GPU/CPU memory habis (OOM). Coba turunkan CONFIG['batch_size']. Error: {e} ---"
        print(error_msg, file=sys.stdout)
        logger.error(error_msg)
        raise
    except Exception as e:
        error_msg = f"--- ERROR: [{model_name}] Training gagal: {type(e).__name__}: {e} ---"
        print(error_msg, file=sys.stdout)
        logger.error(error_msg)
        raise

    if history is None or np.isnan(history.history.get('loss', [0])[-1]):
        logger.warning(
            f"[{model_name}] Loss akhir bernilai NaN atau history kosong -- training tidak stabil. "
            f"Coba turunkan learning rate atau periksa data untuk nilai anomali/corrupt."
        )
        if history is None:
            raise RuntimeError(f"[{model_name}] Training completed, but history object is None. Check logs for earlier errors.")

    logger.info(f"Training '{model_name}' selesai.")
    return history

In [ ]:
def build_mobilenet_model(input_shape, learning_rate=1e-4, resize_to=(96, 96)):
    """
    Bangun model MobileNetV2 dengan base ImageNet, layer klasifikasi custom di atasnya.
    Base model di-freeze dulu -- fine-tuning (unfreeze parsial) dilakukan terpisah.

    resize_to: patch asli (mis. 32x32) di-upscale secara INTERNAL (layer Resizing,
    di dalam graph model) ke resolusi ini sebelum masuk base MobileNetV2 -- pada input
    32x32 mentah, feature map akhir MobileNetV2 jadi 1x1 (informasi spasial nyaris hilang).
    Pada 96x96, feature map akhir jadi 3x3 -- jauh lebih bermakna secara spasial.
    """
    inputs = layers.Input(shape=input_shape)
    x = layers.Resizing(resize_to[0], resize_to[1], interpolation="bilinear",
                         name="upscale_for_mobilenet")(inputs)

    try:
        base_model = MobileNetV2(
            input_shape=(resize_to[0], resize_to[1], input_shape[2]),
            include_top=False, weights='imagenet'
        )
    except Exception as e:
        raise RuntimeError(
            f"Gagal memuat MobileNetV2 pretrained weights: {e}. "
            f"Periksa koneksi internet (weights diunduh dari server Keras saat pertama kali dipanggil)."
        ) from e

    base_model.trainable = False
    x = base_model(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=output)
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model, base_model

In [ ]:
def build_generic_transfer_model(base_model_class, model_name: str, input_shape, learning_rate=1e-4):
    """
    Bangun model transfer learning generik (dipakai untuk VGG16, pola sama dengan MobileNetV2)
    dengan base ImageNet + head klasifikasi custom. Base model di-freeze dulu.

    Raises:
        RuntimeError: jika pretrained weights gagal dimuat (mis. masalah koneksi internet)
    """
    try:
        base_model = base_model_class(input_shape=input_shape, include_top=False, weights='imagenet')
    except Exception as e:
        raise RuntimeError(
            f"[{model_name}] Gagal memuat pretrained weights ImageNet: {e}. "
            f"Periksa koneksi internet (weights diunduh saat pertama kali dipanggil)."
        ) from e

    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model, base_model


def unfreeze_and_recompile(model, base_model, model_name: str, num_unfreeze_layers: int,
                            fine_tune_lr: float, logger: logging.Logger):
    """
    Unfreeze N layer terakhir base model untuk fine-tuning, lalu recompile dengan LR lebih kecil.

    PENTING: layer BatchNormalization TETAP dibekukan (trainable=False) meski berada di rentang
    N layer terakhir -- BatchNorm yang ikut dilatih ulang pada dataset kecil/batch kecil bisa
    merusak running mean/variance hasil kalibrasi ImageNet dan menyebabkan model kolaps.
    """
    if num_unfreeze_layers > len(base_model.layers):
        logger.warning(
            f"[{model_name}] num_unfreeze_layers ({num_unfreeze_layers}) melebihi jumlah layer base "
            f"model ({len(base_model.layers)}) -- semua layer akan di-unfreeze."
        )
        num_unfreeze_layers = len(base_model.layers)

    bn_frozen_count = 0
    for layer in base_model.layers[-num_unfreeze_layers:]:
        if isinstance(layer, BatchNormalization):
            layer.trainable = False
            bn_frozen_count += 1
        else:
            layer.trainable = True

    model.compile(optimizer=Adam(learning_rate=fine_tune_lr),
                   loss='binary_crossentropy', metrics=['accuracy'])
    logger.info(f"[{model_name}] {num_unfreeze_layers} layer terakhir di-unfreeze untuk fine-tuning "
                f"({bn_frozen_count} layer BatchNormalization tetap dibekukan).")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

def evaluate_keras_model(model, model_name: str, val_generator, target_names: list, logger: logging.Logger):
    """Evaluasi model Keras (Custom CNN / MobileNetV2 / VGG16): classification_report + confusion matrix."""
    try:
        val_generator.reset()
        predictions = model.predict(val_generator, verbose=0)
        y_true = val_generator.classes
        y_pred = (predictions > 0.5).astype(int).flatten()
    except Exception as e:
        raise RuntimeError(f"[{model_name}] Evaluasi gagal: {type(e).__name__}: {e}") from e

    report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    report["cohen_kappa"] = kappa
    logger.info(f"[{model_name}] Accuracy: {report['accuracy']:.4f}, "
                f"Macro F1: {report['macro avg']['f1-score']:.4f}, "
                f"Kappa: {kappa:.4f}")
    return report, cm, y_true, y_pred


def evaluate_yolo_model(yolo_model, yolo_dataset_dir: str, class_names: list, logger: logging.Logger):
    """Evaluasi model YOLOv8-cls pada folder val/ -- iterasi manual karena beda API dari Keras."""
    val_dir = os.path.join(yolo_dataset_dir, "val")
    if not os.path.isdir(val_dir):
        raise FileNotFoundError(f"Folder val YOLOv8 tidak ditemukan: '{val_dir}'.")

    y_true, y_pred = [], []
    class_to_idx = {name: i for i, name in enumerate(class_names)}

    for class_name in class_names:
        class_dir = os.path.join(val_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            fpath = os.path.join(class_dir, fname)
            try:
                result = yolo_model.predict(fpath, verbose=False)[0]
                pred_class_idx = int(result.probs.top1)
            except Exception as e:
                logger.warning(f"Gagal prediksi '{fpath}': {e}")
                continue
            y_true.append(class_to_idx[class_name])
            y_pred.append(pred_class_idx)

    if not y_true:
        raise RuntimeError("Evaluasi YOLOv8 gagal total -- tidak ada satupun prediksi berhasil.")

    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    report["cohen_kappa"] = kappa
    logger.info(f"[YOLOv8-cls] Accuracy: {report['accuracy']:.4f}, "
                f"Macro F1: {report['macro avg']['f1-score']:.4f}, "
                f"Kappa: {kappa:.4f}")
    return report, cm, y_true, y_pred

In [ ]:
# ==========================================
# IMPORT ULTRALYTICS (YOLOv8) -- dengan fallback instalasi
# ==========================================
try:
    from ultralytics import YOLO
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics", "--break-system-packages"], check=True)
    from ultralytics import YOLO

import torch

device_global = "cuda" if torch.cuda.is_available() else "cpu"
if device_global == "cpu":
    logger.warning("YOLOv8 akan training di CPU -- proses bisa jauh lebih lambat, apalagi untuk 5 fold.")

## 5. Split Data (Single Train/Val Split)

In [ ]:
def split_and_oversample(df: pd.DataFrame, config: dict, logger: logging.Logger):
    """
    Split stratified train/val, lalu oversample kelas minoritas HANYA di training set
    (val set tetap representasi distribusi asli -- jangan oversample val, itu akan bias evaluasi).

    Raises:
        ValueError: jika salah satu kelas kosong setelah split (dataset terlalu kecil / tidak seimbang ekstrem)
    """
    y_col = config["y_col"]

    try:
        df_train, df_val = train_test_split(
            df, test_size=config["test_size"], stratify=df[y_col], random_state=config["random_state"]
        )
    except ValueError as e:
        raise ValueError(
            f"Gagal melakukan stratified split: {e}. "
            f"Kemungkinan salah satu kelas terlalu sedikit sampelnya untuk di-split dengan stratifikasi."
        ) from e

    minority_class = config["minority_class"]
    minority_df = df_train[df_train[y_col] == minority_class]
    majority_count = df_train[y_col].value_counts().max()
    minority_count = len(minority_df)

    if minority_count == 0:
        raise ValueError(f"Kelas minoritas '{minority_class}' tidak ditemukan di training set setelah split.")

    oversample_factor = max(1, int(np.ceil(majority_count / minority_count)) - 1)
    df_minority_oversampled = pd.concat([minority_df] * oversample_factor, ignore_index=True)
    df_train_final = pd.concat([df_train, df_minority_oversampled], ignore_index=True)
    df_train_final = df_train_final.sample(frac=1, random_state=config["random_state"]).reset_index(drop=True)

    logger.info(f"Split selesai: train={len(df_train_final)} (setelah oversampling), val={len(df_val)}")
    return df_train_final, df_val


df_train, df_val = split_and_oversample(df, CONFIG, logger)

In [ ]:
train_generator, val_generator = build_generators(df_train, df_val, IMAGES_DIR, CONFIG, logger)
class_weights_dict = compute_class_weights_dict(train_generator, logger)
target_names = CONFIG["class_names"]

## 6. Training & Evaluasi 4 Model

### CATATAN: re-seeding + patience Custom CNN

Klaim reproducibility di naskah (`tf.random.set_seed` + `np.random.seed`) memang SUDAH benar diterapkan sejak awal (di cell CONFIG) -- ini bukan bug yang hilang. Tapi seed di awal notebook hanya menabur random state TF sekali; operasi acak apa pun di antara cell itu dan sel training Custom CNN (termasuk akibat sel yang sempat dijalankan ulang di sesi yang sama) bisa menggeser urutan angka acak yang diterima inisialisasi bobot Custom CNN -- satu-satunya dari keempat model yang dilatih from scratch, sehingga paling sensitif terhadap ini. Sel di bawah menambahkan re-seed tepat sebelum Custom CNN dibangun, dan menaikkan `patience` EarlyStopping khusus untuknya dari 10 ke 20.

In [ ]:
input_shape = (CONFIG["img_size"][0], CONFIG["img_size"][1], 3)

# --- Model 1: Custom CNN ---
# Re-seed TEPAT SEBELUM membangun Custom CNN: seed di CONFIG (cell awal) sudah menabur
# random state TF sekali di awal notebook, tapi operasi apa pun yang memakai angka acak di
# antara cell itu dan sini (termasuk sel yang sempat dijalankan ulang di sesi yang sama)
# bisa menggeser urutan angka acak yang diterima inisialisasi bobot Custom CNN. Re-seed di
# sini membuat inisialisasi Custom CNN tidak bergantung pada riwayat eksekusi sebelumnya.
tf.random.set_seed(CONFIG["random_state"])
np.random.seed(CONFIG["random_state"])

custom_cnn_model = build_custom_cnn(input_shape)
custom_cnn_history = train_keras_model(
    custom_cnn_model, "Custom CNN", train_generator, val_generator, CONFIG, class_weights_dict,
    logger, patience=20,  # dinaikkan dari default 10 -- beri ruang lebih untuk lolos dari
                          # dataran rendah di awal training sebelum EarlyStopping aktif
)
cnn_save_path = os.path.join(CONFIG["save_dir"], "custom_cnn_model.keras")
custom_cnn_model.save(cnn_save_path)
logger.info(f"Custom CNN disimpan ke: {cnn_save_path}")


In [ ]:
# --- Model 2: MobileNetV2 (dengan fine-tuning parsial) ---
mobilenet_model, mobilenet_base = build_mobilenet_model(input_shape)
mobilenet_history = train_keras_model(
    mobilenet_model, "MobileNetV2 (frozen)", train_generator, val_generator, CONFIG, class_weights_dict, logger
)

unfreeze_fraction = 0.2
num_unfreeze = max(20, int(len(mobilenet_base.layers) * unfreeze_fraction))
num_unfreeze = min(num_unfreeze, len(mobilenet_base.layers))
for layer in mobilenet_base.layers[-num_unfreeze:]:
    if isinstance(layer, BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True
mobilenet_model.compile(optimizer=Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])

mobilenet_history_finetune = train_keras_model(
    mobilenet_model, "MobileNetV2 (fine-tuned)", train_generator, val_generator, CONFIG, class_weights_dict, logger
)

mn_save_path = os.path.join(CONFIG["save_dir"], "mobilenet_model.keras")
mobilenet_model.save(mn_save_path)
logger.info(f"MobileNetV2 disimpan ke: {mn_save_path}")

In [ ]:
# --- Model 3: VGG16 (dengan fine-tuning) ---
vgg_model, vgg_base = build_generic_transfer_model(VGG16, "VGG16", input_shape)
vgg_history_frozen = train_keras_model(
    vgg_model, "VGG16 (frozen)", train_generator, val_generator, CONFIG, class_weights_dict, logger
)

unfreeze_and_recompile(vgg_model, vgg_base, "VGG16", num_unfreeze_layers=0, fine_tune_lr=1e-4, logger=logger)
vgg_history = train_keras_model(
    vgg_model, "VGG16 (fine-tuned)", train_generator, val_generator, CONFIG, class_weights_dict, logger
)

vgg_save_path = os.path.join(CONFIG["save_dir"], "vgg16_model.keras")
vgg_model.save(vgg_save_path)
logger.info(f"VGG16 disimpan ke: {vgg_save_path}")

### Visualisasi kurva training/validasi (Custom CNN, MobileNetV2, VGG16)

Ditempatkan di sini (setelah ketiga model Keras selesai dilatih, sebelum YOLOv8-cls) karena hanya bergantung pada objek `history` dari ketiga model tersebut. Untuk MobileNetV2 dan VGG16, kurva fase *frozen* dan *fine-tuned* digabung jadi satu garis kontinu supaya terlihat transisinya.

In [ ]:
# --- Kurva Training/Validation Accuracy dan Loss (Custom CNN, MobileNetV2, VGG16) ---
# YOLOv8-cls tidak disertakan di sini karena dilatih lewat framework terpisah (Ultralytics)
# dengan format history yang berbeda; kurva/metrik trainingnya tersimpan otomatis oleh
# Ultralytics di save_dir (results.csv, results.png) dan dievaluasi terpisah di Bagian 8.

def concat_history(*histories, key):
    """Gabungkan beberapa fase training (mis. frozen + fine-tuned) jadi satu kurva kontinu."""
    values = []
    for h in histories:
        if h is not None and key in h.history:
            values.extend(h.history[key])
    return values

curves = {
    "Custom CNN": {
        "acc": concat_history(custom_cnn_history, key="accuracy"),
        "val_acc": concat_history(custom_cnn_history, key="val_accuracy"),
        "loss": concat_history(custom_cnn_history, key="loss"),
        "val_loss": concat_history(custom_cnn_history, key="val_loss"),
    },
    "MobileNetV2": {
        "acc": concat_history(mobilenet_history, mobilenet_history_finetune, key="accuracy"),
        "val_acc": concat_history(mobilenet_history, mobilenet_history_finetune, key="val_accuracy"),
        "loss": concat_history(mobilenet_history, mobilenet_history_finetune, key="loss"),
        "val_loss": concat_history(mobilenet_history, mobilenet_history_finetune, key="val_loss"),
    },
    "VGG16": {
        "acc": concat_history(vgg_history_frozen, vgg_history, key="accuracy"),
        "val_acc": concat_history(vgg_history_frozen, vgg_history, key="val_accuracy"),
        "loss": concat_history(vgg_history_frozen, vgg_history, key="loss"),
        "val_loss": concat_history(vgg_history_frozen, vgg_history, key="val_loss"),
    },
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {"Custom CNN": "tab:blue", "MobileNetV2": "tab:green", "VGG16": "tab:red"}

for model_name, c in curves.items():
    epochs_range = range(1, len(c["acc"]) + 1)
    axes[0].plot(epochs_range, c["acc"], linestyle="--", color=colors[model_name],
                 label=f"{model_name} Train Acc")
    axes[0].plot(epochs_range, c["val_acc"], linestyle="-", color=colors[model_name],
                 label=f"{model_name} Val Acc")
    axes[1].plot(epochs_range, c["loss"], linestyle="--", color=colors[model_name],
                 label=f"{model_name} Train Loss")
    axes[1].plot(epochs_range, c["val_loss"], linestyle="-", color=colors[model_name],
                 label=f"{model_name} Val Loss")

axes[0].set_title("Accuracy Comparison: 3 Models")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_title("Loss Comparison: 3 Models")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()

fig_save_path = os.path.join(CONFIG["save_dir"], "training_curves_accuracy_loss.png")
plt.savefig(fig_save_path, dpi=300, bbox_inches="tight")
logger.info(f"Kurva accuracy/loss disimpan ke: {fig_save_path}")

plt.show()


### CATATAN REVISI v3 YOLO (bug struktural: file duplikat saling menimpa)

Oversampling standar tidak pernah benar-benar sampai ke folder YOLO sejak notebook asli (baris duplikat = nama file sama = saling menimpa). v3 memberi nama tujuan unik per baris dan mencetak jumlah file fisik aktual di akhir sel untuk verifikasi.

In [ ]:
# --- Model 4: YOLOv8-cls ---
def prepare_yolo_folder_structure_weighted(df_train, df_val, images_dir, yolo_dataset_dir,
                                             config, class_weights_dict, logger):
    """
    Susun folder train/val/<kelas>/ untuk format YOLOv8-cls, DENGAN oversampling ekstra
    khusus YOLO -- kompensasi karena Ultralytics tidak mendukung class-weighted loss native.

    versi ke-3, memperbaiki BUG STRUKTURAL yang ternyata sudah ada sejak notebook ASLI: baris
    hasil oversampling di df_train_final adalah baris DUPLIKAT dengan nama file yang SAMA
    PERSIS. Fungsi penyalinan lama memakai nama file asli sebagai nama tujuan -- setiap
    duplikat MENIMPA file tujuan yang sama, bukan membuat file baru. Akibatnya folder YOLO
    hanya pernah berisi file UNIK, sehingga oversampling standar TIDAK PERNAH benar-benar
    sampai ke YOLO sama sekali -- bahkan sebelum revisi manapun.

    Perbaikan v3: setiap file disalin dengan nama tujuan yang UNIK per baris (suffix _copyN
    untuk baris duplikat ke-2 dst.), sehingga oversampling standar maupun oversampling ekstra
    tambahan benar-benar menjadi file fisik terpisah, bukan saling menimpa.

    Catatan metodologis: pendekatan oversampling ekstra ini tetap MENDEKATI (approximation)
    terhadap efek class-weighted loss di Keras, bukan penyetaraan matematis persis.
    """
    if os.path.exists(yolo_dataset_dir):
        shutil.rmtree(yolo_dataset_dir)

    x_col, y_col = config["x_col"], config["y_col"]
    class_names = config["class_names"]

    boost_idx = max(class_weights_dict, key=class_weights_dict.get)
    boost_weight = class_weights_dict[boost_idx]
    boost_class = class_names[boost_idx]
    logger.info(
        f"[YOLO] Kelas yang didorong ekstra: '{boost_class}' (class_weight={boost_weight:.4f}). "
        f"Ditentukan dari bobot aktual, bukan dari config['minority_class']."
    )

    seen_counts = {}

    def copy_with_unique_name(src, dst_dir, fname):
        """Salin file dengan nama tujuan unik -- baris duplikat (nama file sama, dari
        oversampling) tidak akan saling menimpa satu sama lain."""
        stem, ext = os.path.splitext(fname)
        key = (dst_dir, fname)
        seen_counts[key] = seen_counts.get(key, 0) + 1
        count = seen_counts[key]
        dst_name = fname if count == 1 else f"{stem}_copy{count - 1}{ext}"
        shutil.copy(src, os.path.join(dst_dir, dst_name))

    for split_name, split_df in [("train", df_train), ("val", df_val)]:
        for class_name in class_names:
            os.makedirs(os.path.join(yolo_dataset_dir, split_name, class_name), exist_ok=True)

        for _, row in split_df.iterrows():
            fname, cls = row[x_col], row[y_col]
            src = os.path.join(images_dir, fname)
            if not os.path.exists(src):
                continue
            dst_dir = os.path.join(yolo_dataset_dir, split_name, cls)
            copy_with_unique_name(src, dst_dir, fname)

        if split_name == "train" and boost_weight > 1.0:
            class_df = split_df[split_df[y_col] == boost_class].reset_index(drop=True)
            n_extra = int(round((boost_weight - 1.0) * len(class_df)))
            logger.info(
                f"[YOLO] Menambah {n_extra} salinan acak dari kelas '{boost_class}' "
                f"(dari total {len(class_df)} baris kelas ini di train) sebagai kompensasi bobot."
            )
            if n_extra > 0:
                extra_rows = class_df.sample(n=n_extra, replace=(n_extra > len(class_df)),
                                              random_state=config["random_state"])
                dst_dir = os.path.join(yolo_dataset_dir, split_name, boost_class)
                for _, row in extra_rows.iterrows():
                    fname = row[x_col]
                    src = os.path.join(images_dir, fname)
                    if os.path.exists(src):
                        copy_with_unique_name(src, dst_dir, fname)

    logger.info(f"Struktur folder YOLOv8 (oversampling terkoreksi, file unik) siap di: {yolo_dataset_dir}")

    for split_name in ["train", "val"]:
        for class_name in class_names:
            folder = os.path.join(yolo_dataset_dir, split_name, class_name)
            n_files = len(os.listdir(folder)) if os.path.isdir(folder) else 0
            logger.info(f"[YOLO] Cek folder {split_name}/{class_name}: {n_files} file fisik")


YOLO_DATASET_DIR = CONFIG["yolo_dataset_dir"]
prepare_yolo_folder_structure_weighted(
    df_train, df_val, IMAGES_DIR, YOLO_DATASET_DIR, CONFIG, class_weights_dict, logger
)

yolo_model = YOLO(CONFIG["yolo_model_arch"])
yolo_model.train(
    data=YOLO_DATASET_DIR, epochs=CONFIG["yolo_epochs"], imgsz=CONFIG["yolo_imgsz"],
    batch=CONFIG["yolo_batch"], lr0=CONFIG["yolo_lr0"], lrf=CONFIG["yolo_lrf"],
    cos_lr=True, warmup_epochs=3, patience=CONFIG["yolo_patience"], dropout=0.2,
    degrees=15.0, flipud=0.3, fliplr=0.5,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    device=device_global, project="runs_classify", name="yolov8cls_integrated", exist_ok=True,
    verbose=False,
)


### Kurva training khusus YOLOv8-cls

Dipisah dari plot 3 model Keras karena Ultralytics mencatat metrik secara berbeda (tidak ada train accuracy per-epoch, hanya val accuracy top1/top5). Sel ini membaca `results.csv` bawaan Ultralytics langsung dari direktori training, sekaligus menyalinnya ke Google Drive supaya tidak hilang kalau runtime Colab terputus.

In [ ]:
# --- Kurva Training Khusus YOLOv8-cls ---
# Terpisah dari plot 3 model Keras di atas, karena Ultralytics mencatat metrik dengan cara
# berbeda: tidak ada "train accuracy" per-epoch seperti Keras (hanya top1/top5 accuracy pada
# val set), sehingga memaksakannya ke plot yang sama akan menghasilkan bentuk yang tidak sepadan.

yolo_results_path = os.path.join(yolo_model.trainer.save_dir, "results.csv")
print("Path results YOLO:", yolo_results_path)
print("Ada:", os.path.exists(yolo_results_path))

if os.path.exists(yolo_results_path):
    # Salin ke Drive juga supaya tidak hilang kalau runtime Colab terputus
    yolo_csv_backup = os.path.join(CONFIG["save_dir"], "yolo_results.csv")
    shutil.copy(yolo_results_path, yolo_csv_backup)
    logger.info(f"results.csv YOLOv8-cls disalin ke: {yolo_csv_backup}")

    yolo_df = pd.read_csv(yolo_results_path)
    yolo_df.columns = [c.strip() for c in yolo_df.columns]
    print("Kolom tersedia:", yolo_df.columns.tolist())

    # Nama kolom standar Ultralytics classification; fallback dicek kalau versi berbeda
    def pick_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return None

    col_epoch = pick_col(yolo_df, ["epoch"])
    col_train_loss = pick_col(yolo_df, ["train/loss"])
    col_val_loss = pick_col(yolo_df, ["val/loss"])
    col_top1 = pick_col(yolo_df, ["metrics/accuracy_top1"])

    missing = [n for n, c in [("epoch", col_epoch), ("train/loss", col_train_loss),
                               ("val/loss", col_val_loss), ("accuracy_top1", col_top1)] if c is None]
    if missing:
        print("PERINGATAN: kolom berikut tidak ditemukan, cek nama kolom di atas dan sesuaikan "
              f"fungsi pick_col: {missing}")
    else:
        fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))

        axes3[0].plot(yolo_df[col_epoch], yolo_df[col_train_loss], linestyle="--",
                      color="tab:purple", label="YOLOv8-cls Train Loss")
        axes3[0].plot(yolo_df[col_epoch], yolo_df[col_val_loss], linestyle="-",
                      color="tab:purple", label="YOLOv8-cls Val Loss")
        axes3[0].set_title("YOLOv8-cls: Loss per Epoch")
        axes3[0].set_xlabel("Epoch")
        axes3[0].set_ylabel("Loss")
        axes3[0].legend(fontsize=9)
        axes3[0].grid(alpha=0.3)

        axes3[1].plot(yolo_df[col_epoch], yolo_df[col_top1], linestyle="-",
                      color="tab:purple", label="YOLOv8-cls Val Accuracy (top1)")
        axes3[1].set_title("YOLOv8-cls: Accuracy (top1) per Epoch")
        axes3[1].set_xlabel("Epoch")
        axes3[1].set_ylabel("Accuracy")
        axes3[1].set_ylim(0, 1.05)
        axes3[1].legend(fontsize=9)
        axes3[1].grid(alpha=0.3)
        axes3[1].annotate("Catatan: Ultralytics tidak mencatat train accuracy per-epoch,\n"
                          "hanya accuracy pada val set (top1/top5).",
                          xy=(0.02, 0.02), xycoords="axes fraction", fontsize=8, color="gray")

        plt.tight_layout()
        yolo_fig_path = os.path.join(CONFIG["save_dir"], "yolo_training_curve.png")
        plt.savefig(yolo_fig_path, dpi=300, bbox_inches="tight")
        logger.info(f"Kurva khusus YOLOv8-cls disimpan ke: {yolo_fig_path}")
        plt.show()
else:
    print("File results.csv tidak ditemukan -- kemungkinan runtime Colab sudah terputus sejak "
          "training YOLOv8-cls terakhir. YOLOv8-cls perlu dilatih ulang (jalankan ulang sel "
          "training YOLOv8-cls di atas) sebelum sel ini bisa menghasilkan kurva.")


In [ ]:
# --- Evaluasi keempat model pada val set yang SAMA ---
cnn_report, cnn_cm, cnn_y_true, cnn_y_pred = evaluate_keras_model(
    custom_cnn_model, "Custom CNN", val_generator, target_names, logger
)
mobilenet_report, mobilenet_cm, mn_y_true, mn_y_pred = evaluate_keras_model(
    mobilenet_model, "MobileNetV2", val_generator, target_names, logger
)
vgg_report, vgg_cm, vgg_y_true, vgg_y_pred = evaluate_keras_model(
    vgg_model, "VGG16", val_generator, target_names, logger
)
yolo_report, yolo_cm, yolo_y_true, yolo_y_pred = evaluate_yolo_model(
    yolo_model, YOLO_DATASET_DIR, CONFIG["class_names"], logger
)

### Visualisasi Confusion Matrix (belum ada di notebook asli)

`cnn_cm`, `mobilenet_cm`, `vgg_cm`, `yolo_cm` sudah dihitung di sel evaluasi di atas, tapi tidak pernah ditampilkan. Sel ini menampilkannya sebagai heatmap (untuk Gambar 3 di naskah) sekaligus mencetak angka TN/FP/FN/TP mentah (untuk Tabel 2).

In [ ]:
# --- Visualisasi Confusion Matrix keempat model (dihitung di sel evaluasi di atas,
#     tapi belum pernah ditampilkan di notebook asli) ---
cm_dict = {
    "Custom CNN": cnn_cm,
    "MobileNetV2": mobilenet_cm,
    "VGG16": vgg_cm,
    "YOLOv8-cls": yolo_cm,
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (model_name, cm) in zip(axes, cm_dict.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CONFIG["class_names"], yticklabels=CONFIG["class_names"], ax=ax)
    ax.set_title(model_name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()

cm_fig_path = os.path.join(CONFIG["save_dir"], "confusion_matrices_4_models.png")
plt.savefig(cm_fig_path, dpi=300, bbox_inches="tight")
logger.info(f"Confusion matrix (4 model) disimpan ke: {cm_fig_path}")

plt.show()

# --- Cetak juga sebagai angka mentah, untuk dicek/disalin langsung ke Tabel 2 ---
for model_name, cm in cm_dict.items():
    tn, fp, fn, tp = cm.ravel()
    print(f"{model_name}: TN={tn}, FP={fp}, FN={fn}, TP={tp}  "
          f"(baris/kolom 0={CONFIG[\'class_names\'][0]}, 1={CONFIG[\'class_names\'][1]})")


## 7. Tabel Perbandingan Akhir -- 4 Model

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Custom CNN", "MobileNetV2", "VGG16", "YOLOv8-cls"],
    "Accuracy": [cnn_report["accuracy"], mobilenet_report["accuracy"],
                 vgg_report["accuracy"], yolo_report["accuracy"]],
    "Precision (Macro)": [cnn_report["macro avg"]["precision"], mobilenet_report["macro avg"]["precision"],
                           vgg_report["macro avg"]["precision"], yolo_report["macro avg"]["precision"]],
    "Recall (Macro)": [cnn_report["macro avg"]["recall"], mobilenet_report["macro avg"]["recall"],
                        vgg_report["macro avg"]["recall"], yolo_report["macro avg"]["recall"]],
    "F1-Score (Macro)": [cnn_report["macro avg"]["f1-score"], mobilenet_report["macro avg"]["f1-score"],
                          vgg_report["macro avg"]["f1-score"], yolo_report["macro avg"]["f1-score"]],
    "Cohen's Kappa": [cnn_report["cohen_kappa"], mobilenet_report["cohen_kappa"],
                       vgg_report["cohen_kappa"], yolo_report["cohen_kappa"]],
})

comparison_save_path = os.path.join(CONFIG["save_dir"], "comparison_single_split.csv")
comparison_df.to_csv(comparison_save_path, index=False)
logger.info(f"Tabel perbandingan disimpan ke: {comparison_save_path}")
display(comparison_df)

## 8. Bootstrap 95% CI + McNemar's Test

**Langkah 1**: gabungkan prediksi keempat model berdasarkan **nama file** (bukan sekadar urutan indeks) --
supaya benar-benar dibandingkan pada sampel yang persis sama, karena YOLOv8-cls mengevaluasi lewat
struktur folder terpisah (urutan berbeda dari `val_generator`).

In [ ]:
import numpy as np

# --- Filenames untuk model Keras (CNN/MobileNetV2/VGG16 berbagi val_generator yang sama) ---
val_filenames = val_generator.filenames

df_keras_preds = pd.DataFrame({
    "filename": val_filenames,
    "y_true": cnn_y_true,
    "pred_cnn": cnn_y_pred,
    "pred_mn": mn_y_pred,
    "pred_vgg": vgg_y_pred,
})

# Sanity check: pastikan urutan y_true identik antar model Keras (seharusnya PASTI sama
# karena berbagi val_generator yang sama persis) -- kalau tidak, ada sesuatu yang salah.
if not np.array_equal(cnn_y_true, mn_y_true):
    raise RuntimeError(
        "y_true Custom CNN dan MobileNetV2 TIDAK identik meski berbagi val_generator yang sama. "
        "Kemungkinan val_generator di-reset/diacak ulang di antara evaluasi -- periksa urutan sel."
    )
if not np.array_equal(cnn_y_true, vgg_y_true):
    raise RuntimeError(
        "y_true Custom CNN dan VGG16 TIDAK identik meski berbagi val_generator yang sama. "
        "Kemungkinan val_generator di-reset/diacak ulang di antara evaluasi -- periksa urutan sel."
    )
logger.info(f"Prediksi Keras (CNN/MobileNetV2/VGG16) selaras: {len(df_keras_preds)} sampel.")

In [ ]:
# --- Prediksi YOLOv8-cls dengan filename eksplisit (evaluasi ulang ringan -- HANYA inference,
#     bukan training ulang, jadi cepat dan murah komputasi) ---
def get_yolo_predictions_with_filenames(yolo_model, yolo_dataset_dir, class_names, logger):
    val_dir = os.path.join(yolo_dataset_dir, "val")
    if not os.path.isdir(val_dir):
        raise FileNotFoundError(f"Folder val YOLOv8 tidak ditemukan: '{val_dir}'.")

    class_to_idx = {name: i for i, name in enumerate(class_names)}
    rows = []
    for class_name in class_names:
        class_dir = os.path.join(val_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            fpath = os.path.join(class_dir, fname)
            try:
                result = yolo_model.predict(fpath, verbose=False)[0]
                pred_idx = int(result.probs.top1)
            except Exception as e:
                logger.warning(f"Gagal prediksi YOLO untuk '{fpath}': {e}")
                continue
            rows.append({"filename": fname, "y_true_yolo": class_to_idx[class_name], "pred_yolo": pred_idx})

    if not rows:
        raise RuntimeError("Tidak ada satupun prediksi YOLO berhasil diambil dengan filename.")
    return pd.DataFrame(rows)


df_yolo_preds = get_yolo_predictions_with_filenames(yolo_model, YOLO_DATASET_DIR, CONFIG["class_names"], logger)
logger.info(f"Prediksi YOLOv8-cls dengan filename: {len(df_yolo_preds)} sampel.")

In [ ]:
# --- Gabungkan berdasarkan filename ---
df_combined = df_keras_preds.merge(df_yolo_preds, on="filename", how="inner")

n_before, n_after = len(df_keras_preds), len(df_combined)
if n_after < n_before:
    logger.warning(
        f"{n_before - n_after} sampel TIDAK ketemu pasangan filename antara Keras dan YOLOv8 -- "
        f"dikeluarkan dari analisis. Kemungkinan penamaan file sedikit berbeda antar folder."
    )
if n_after == 0:
    raise RuntimeError(
        "TIDAK ADA satupun filename yang cocok antara prediksi Keras dan YOLOv8 -- "
        "periksa apakah nama file di val_generator dan folder YOLO val/ benar-benar sama."
    )

mismatch = (df_combined["y_true"] != df_combined["y_true_yolo"]).sum()
if mismatch > 0:
    raise ValueError(
        f"{mismatch} sampel punya label y_true BERBEDA antara Keras dan YOLO untuk filename yang sama -- "
        f"ada masalah alignment data. JANGAN lanjut ke bootstrap/McNemar sebelum ini diperbaiki."
    )

logger.info(f"Data gabungan siap untuk bootstrap & McNemar: {len(df_combined)} sampel, label selaras.")
df_combined.head()

### 8a. Bootstrap Resampling (2000x) -- 95% Confidence Interval

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

N_BOOTSTRAP = 2000
rng = np.random.default_rng(CONFIG["random_state"])

model_pred_cols = {"Custom CNN": "pred_cnn", "MobileNetV2": "pred_mn", "VGG16": "pred_vgg", "YOLOv8-cls": "pred_yolo"}
bootstrap_results = {name: {"accuracy": [], "f1_macro": [], "kappa": []} for name in model_pred_cols}

n = len(df_combined)
y_true_arr = df_combined["y_true"].values
skipped = 0

for b in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, size=n)  # resample DENGAN penggantian
    yt = y_true_arr[idx]
    if len(np.unique(yt)) < 2:
        # resample kebetulan cuma berisi 1 kelas -- tidak valid untuk f1_macro/kappa, lewati
        skipped += 1
        continue
    for name, col in model_pred_cols.items():
        yp = df_combined[col].values[idx]
        bootstrap_results[name]["accuracy"].append(accuracy_score(yt, yp))
        bootstrap_results[name]["f1_macro"].append(f1_score(yt, yp, average="macro", zero_division=0))
        bootstrap_results[name]["kappa"].append(cohen_kappa_score(yt, yp))

if skipped > 0:
    logger.warning(f"{skipped}/{N_BOOTSTRAP} resample dilewati (hanya berisi 1 kelas).")

summary_rows = []
for name, metrics in bootstrap_results.items():
    row = {"Model": name}
    for metric_name, values in metrics.items():
        values = np.array(values)
        row[f"{metric_name}_mean"] = values.mean()
        row[f"{metric_name}_ci_low"] = np.percentile(values, 2.5)
        row[f"{metric_name}_ci_high"] = np.percentile(values, 97.5)
    summary_rows.append(row)

df_bootstrap_summary = pd.DataFrame(summary_rows)
urutan_model = ["Custom CNN", "MobileNetV2", "VGG16", "YOLOv8-cls"]
df_bootstrap_summary["Model"] = pd.Categorical(df_bootstrap_summary["Model"], categories=urutan_model, ordered=True)
df_bootstrap_summary = df_bootstrap_summary.sort_values("Model").reset_index(drop=True)

bootstrap_save_path = os.path.join(CONFIG["save_dir"], "bootstrap_ci_summary.csv")
df_bootstrap_summary.to_csv(bootstrap_save_path, index=False)
logger.info(f"Ringkasan bootstrap 95% CI disimpan ke: {bootstrap_save_path}")
display(df_bootstrap_summary)

### 8b. McNemar's Test -- Signifikansi Perbedaan Antar Model

In [ ]:
try:
    from statsmodels.stats.contingency_tables import mcnemar
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "statsmodels", "--break-system-packages"], check=True)
    from statsmodels.stats.contingency_tables import mcnemar

from itertools import combinations


def mcnemar_test_pair(y_true, pred_a, pred_b):
    """Uji McNemar untuk satu pasang model -- membandingkan apakah pola benar/salah
    keduanya berbeda secara signifikan pada sampel yang SAMA."""
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    both_correct = int(np.sum(correct_a & correct_b))
    only_a = int(np.sum(correct_a & ~correct_b))
    only_b = int(np.sum(~correct_a & correct_b))
    both_wrong = int(np.sum(~correct_a & ~correct_b))
    table = [[both_correct, only_a], [only_b, both_wrong]]

    # exact test dipakai kalau jumlah sel "tidak sepakat" (only_a + only_b) kecil (<25) --
    # rekomendasi umum karena chi-square asimtotik tidak akurat untuk sampel kecil.
    use_exact = (only_a + only_b) < 25
    result = mcnemar(table, exact=use_exact, correction=not use_exact)
    return table, result.statistic, result.pvalue, only_a, only_b


model_names = list(model_pred_cols.keys())
mcnemar_rows = []
for name_a, name_b in combinations(model_names, 2):
    col_a, col_b = model_pred_cols[name_a], model_pred_cols[name_b]
    table, stat, pvalue, only_a, only_b = mcnemar_test_pair(
        df_combined["y_true"].values, df_combined[col_a].values, df_combined[col_b].values
    )
    mcnemar_rows.append({
        "Model A": name_a, "Model B": name_b,
        "Hanya A Benar": only_a, "Hanya B Benar": only_b,
        "Statistic": round(stat, 4), "p-value": round(pvalue, 4),
        "Signifikan (p<0.05)": "Ya" if pvalue < 0.05 else "Tidak",
    })

df_mcnemar = pd.DataFrame(mcnemar_rows)
mcnemar_save_path = os.path.join(CONFIG["save_dir"], "mcnemar_test_results.csv")
df_mcnemar.to_csv(mcnemar_save_path, index=False)
logger.info(f"Hasil McNemar\'s test disimpan ke: {mcnemar_save_path}")
display(df_mcnemar)

## 9. Ringkasan Siap-Tempel untuk Artikel

In [ ]:
print("=== Bootstrap 95% CI (2000 resample) ===\n")
for _, row in df_bootstrap_summary.iterrows():
    print(f"{row['Model']}:")
    print(f"  Accuracy : {row['accuracy_mean']:.4f}  [95% CI: {row['accuracy_ci_low']:.4f} - {row['accuracy_ci_high']:.4f}]")
    print(f"  F1 (M)   : {row['f1_macro_mean']:.4f}  [95% CI: {row['f1_macro_ci_low']:.4f} - {row['f1_macro_ci_high']:.4f}]")
    print(f"  Kappa    : {row['kappa_mean']:.4f}  [95% CI: {row['kappa_ci_low']:.4f} - {row['kappa_ci_high']:.4f}]")
    print()

print("\n=== McNemar\'s Test (signifikansi perbedaan antar model) ===\n")
for _, row in df_mcnemar.iterrows():
    kesimpulan = (
        f"perbedaan performa SIGNIFIKAN (p={row['p-value']:.4f})" if row["Signifikan (p<0.05)"] == "Ya"
        else f"perbedaan performa TIDAK signifikan (p={row['p-value']:.4f}) -- bisa jadi kebetulan sampel kecil"
    )
    print(f"{row['Model A']} vs {row['Model B']}: {kesimpulan}")